In [22]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader, TensorDataset

In [26]:
dataset = load_dataset('glue', 'sst2')

# 학습 데이터
train_sentences = list(dataset['train']['sentence'][:500])
train_labels = list(dataset['train']['label'][:500])

# 검증 데이터
val_sentences = dataset['validation']['sentence'][:500]
val_labels = dataset['validation']['label'][:500]

In [27]:
# 기본 토큰나이저와 모델 생성
# 나중에 문장들을 토큰화시키기 위함과 모델을 사용하기 위함
num_labels = 2

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_labels)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [28]:
inputs = tokenizer(train_sentences, return_tensors='pt', padding=True, truncation=True)

dataset_torch = TensorDataset(
    inputs['input_ids'],        # 토큰화된 문장
    inputs['attention_mask'],   # 패딩 구분
    torch.tensor(train_labels)  # 레이블
)

loader = DataLoader(dataset_torch, batch_size=32, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
epoch = 0
for epoch in range(3):          # 전체 데이터 3번 반복
    for input_ids, attention_mask, labels in loader:  # 배치씩
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids,
                       attention_mask=attention_mask,
                       labels=labels)
        outputs.loss.backward()
        optimizer.step()
    print(f"Epoch {epoch}, Loss: {outputs.loss.item():.4f}")

Epoch 0, Loss: 0.6138
Epoch 1, Loss: 0.2200
Epoch 2, Loss: 0.0760


In [29]:
model.eval()

# validation 데이터로 예측
val_inputs = tokenizer(list(val_sentences[:20]), 
                       return_tensors='pt', 
                       padding=True, 
                       truncation=True)

with torch.no_grad():
    result = model(input_ids=val_inputs['input_ids'],
                   attention_mask=val_inputs['attention_mask'])
    pred = result.logits.argmax(dim=1)
    
for sentence, p, label in zip(val_sentences[:20], pred, val_labels[:20]):
    print(f"{sentence[:30]}... → 예측: {'긍정' if p==1 else '부정'} | 정답: {'긍정' if label==1 else '부정'}")

it 's a charming and often aff... → 예측: 긍정 | 정답: 긍정
unflinchingly bleak and desper... → 예측: 부정 | 정답: 부정
allows us to hope that nolan i... → 예측: 긍정 | 정답: 긍정
the acting , costumes , music ... → 예측: 긍정 | 정답: 긍정
it 's slow -- very , very slow... → 예측: 부정 | 정답: 부정
although laced with humor and ... → 예측: 긍정 | 정답: 긍정
a sometimes tedious film . ... → 예측: 부정 | 정답: 부정
or doing last year 's taxes wi... → 예측: 부정 | 정답: 부정
you do n't have to know about ... → 예측: 긍정 | 정답: 긍정
in exactly 89 minutes , most o... → 예측: 부정 | 정답: 부정
the mesmerizing performances o... → 예측: 긍정 | 정답: 긍정
it takes a strange kind of laz... → 예측: 부정 | 정답: 부정
... the film suffers from a la... → 예측: 부정 | 정답: 부정
we root for ( clara and paul )... → 예측: 긍정 | 정답: 긍정
even horror fans will most lik... → 예측: 부정 | 정답: 부정
a gorgeous , high-spirited mus... → 예측: 긍정 | 정답: 긍정
the emotions are raw and will ... → 예측: 긍정 | 정답: 긍정
audrey tatou has a knack for p... → 예측: 긍정 | 정답: 긍정
... the movie is just a plain ... → 예측: 부정 | 정답: 부정
in its best mom